# Select Few-Shot Examples for Endoscapes CVS

Shows all labeled endoscapes annotations with images and rubric details so we can decide which to use as few-shot examples.

In [ ]:
from __future__ import annotations

import base64
import json
import mimetypes
import random
import shutil
from collections import Counter
from pathlib import Path
from typing import Dict, Iterable, List

import pandas as pd
from IPython.display import HTML, display

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'rubrics/cvs_rubrics_v3.json').is_file())

ANNOTATIONS_PATH = (
    REPO_ROOT
    / 'annotations'
    / 'rubric_labels'
    / 'endoscapes_val__rubrics_v1__seed13__batch0__filtered_no_all_uncertain_fix_error.jsonl'
)

LABELS_ROOT = REPO_ROOT / 'data' / 'endoscapes'
SPLIT = 'val'
LABELS_PATH = LABELS_ROOT / SPLIT / 'annotation_ds_coco.json'

RUBRIC_PATH = REPO_ROOT / 'rubrics' / 'cvs_rubrics_v3.json'

print('ANNOTATIONS_PATH:', ANNOTATIONS_PATH)
print('LABELS_PATH:', LABELS_PATH)
print('RUBRIC_PATH:', RUBRIC_PATH)

In [ ]:
def iter_jsonl(path: Path) -> Iterable[dict]:
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def file_key_from_path(image_path: str | Path) -> str:
    return Path(image_path).name


def status_to_bin(status: str | None) -> int:
    if status == 'yes':
        return 1
    return 0


def gt_pattern(gt: Dict[str, int]) -> str:
    return f"{int(gt['c1'])}{int(gt['c2'])}{int(gt['c3'])}"


def img_tag(path: str | Path | None, *, width: int = 300) -> str:
    if not path:
        return ''
    p = Path(path)
    try:
        p = p.expanduser().resolve()
    except Exception:
        p = Path(path)
    if not p.exists():
        return f"<div style='color:#a00'>missing: {p}</div>"
    mime = mimetypes.guess_type(str(p))[0] or 'image/jpeg'
    data = base64.b64encode(p.read_bytes()).decode('ascii')
    return f"<img src='data:{mime};base64,{data}' style='max-width:{width}px' />"


def fmt_rubric_labels(rubrics: Dict[str, str], rubric_items: Dict[str, dict]) -> str:
    """Render rubric labels as a colored HTML table grouped by criterion."""
    color_map = {'yes': '#2ca02c', 'no': '#d62728', 'uncertain': '#ff7f0e'}
    html = '<table style="font-size:11px;border-collapse:collapse;margin:2px 0">'
    html += '<tr><th>ID</th><th>Answer</th><th>Question</th></tr>'
    for item_id in sorted(rubrics.keys()):
        ans = rubrics[item_id]
        color = color_map.get(ans, '#333')
        text = rubric_items.get(item_id, {}).get('text', '')
        html += (
            f'<tr>'
            f'<td style="font-weight:bold;padding:1px 4px">{item_id}</td>'
            f'<td style="color:{color};font-weight:bold;padding:1px 4px">{ans}</td>'
            f'<td style="padding:1px 4px">{text}</td>'
            f'</tr>'
        )
    html += '</table>'
    return html


# Load rubric item definitions
rubric_data = json.loads(RUBRIC_PATH.read_text())
RUBRIC_ITEMS = {}
for crit, block in rubric_data['criteria'].items():
    for item in block['items']:
        RUBRIC_ITEMS[item['id']] = item

print(f'Loaded {len(RUBRIC_ITEMS)} rubric items')

In [ ]:
# -----------------------------
# Load ground-truth criterion labels (continuous -> binary)
# -----------------------------
if not LABELS_PATH.exists():
    raise FileNotFoundError(f'Missing labels file: {LABELS_PATH}')

labels_data = json.loads(LABELS_PATH.read_text())
gt_criterion_by_file: Dict[str, Dict[str, int]] = {}
for img in labels_data.get('images', []):
    file_name = img.get('file_name')
    ds = img.get('ds')
    if not file_name or not isinstance(ds, list) or len(ds) < 3:
        continue
    try:
        c1 = float(ds[0])
        c2 = float(ds[1])
        c3 = float(ds[2])
    except Exception:
        continue
    gt_criterion_by_file[file_name] = {
        'c1': 1 if c1 > 0.5 else 0,
        'c2': 1 if c2 > 0.5 else 0,
        'c3': 1 if c3 > 0.5 else 0,
    }

print('GT criterion rows:', len(gt_criterion_by_file))


In [ ]:
# -----------------------------
# Load rubric labels + metadata
# -----------------------------
if not ANNOTATIONS_PATH.exists():
    raise FileNotFoundError(f'Missing annotations file: {ANNOTATIONS_PATH}')

all_rows = []
for row in iter_jsonl(ANNOTATIONS_PATH):
    image = row.get('image', {})
    file_name = image.get('file_name') or file_key_from_path(image.get('image_path', ''))
    if not file_name:
        continue
    image_path = (LABELS_ROOT / SPLIT / file_name).as_posix()
    if not image_path and file_name:
        image_path = (LABELS_ROOT / SPLIT / file_name).as_posix()
    rubrics = row.get('rubrics') or row.get('rubric_labels') or {}
    gt = gt_criterion_by_file.get(file_name)
    if not gt:
        continue
    all_rows.append({
        'file_name': file_name,
        'image_path': image_path,
        'video_id': str(image.get('video_id', '')),
        'frame_id': image.get('frame_id'),
        'split': row.get('split') or image.get('split') or SPLIT,
        'gt': gt,
        'rubrics': rubrics,
    })

print(f'Total annotations: {len(all_rows)}')

gts = Counter(gt_pattern(r['gt']) for r in all_rows)
print(f'\nBy GT pattern:')
for g, c in sorted(gts.items()):
    print(f'  {g}: {c}')

vids = Counter(r['video_id'] for r in all_rows)
print(f'\nBy video_id ({len(vids)} videos):')
for v, c in sorted(vids.items(), key=lambda x: int(x[0]) if x[0].isdigit() else 0):
    print(f'  {v}: {c}')

In [ ]:
# Display all examples grouped by GT pattern
all_rows_sorted = sorted(all_rows, key=lambda r: (gt_pattern(r['gt']), r['video_id'], r.get('file_name', '')))

for idx, r in enumerate(all_rows_sorted):
    gt = r['gt']
    rubrics = r.get('rubrics', {})
    gp = gt_pattern(gt)
    vid = r['video_id']
    fid = r.get('frame_id', '')
    img_path = str(LABELS_ROOT / 'val' / r['file_name'])

    n_yes = sum(1 for v in rubrics.values() if v == 'yes')
    n_no = sum(1 for v in rubrics.values() if v == 'no')
    n_unc = sum(1 for v in rubrics.values() if v == 'uncertain')

    # Color GT values
    gt_parts = []
    for c in ['c1', 'c2', 'c3']:
        v = gt.get(c, 0)
        color = '#2ca02c' if v == 1 else '#d62728'
        gt_parts.append(f'<span style="color:{color};font-weight:bold">{c.upper()}={v}</span>')
    gt_html = ' &nbsp; '.join(gt_parts)

    html = f'''
    <div style="border:1px solid #ddd;padding:10px;margin:8px 0;display:flex;gap:15px;align-items:flex-start">
        <div style="flex-shrink:0">
            <div style="font-weight:bold;font-size:13px;margin-bottom:4px">
                #{idx+1} &nbsp; GT: {gp} &nbsp;
                <span style="color:#95a5a6;font-size:12px">[video {vid}]</span>
            </div>
            {img_tag(img_path, width=350)}
            <div style="font-size:11px;color:#666;margin-top:4px">
                video: {vid}<br>
                file: {r.get('file_name', '')}
            </div>
        </div>
        <div style="flex:1">
            <div style="margin-bottom:6px">
                <b>GT:</b> {gt_html} &nbsp;&nbsp;
                <b>Rubrics:</b> <span style="color:#2ca02c">{n_yes} yes</span>,
                <span style="color:#d62728">{n_no} no</span>,
                <span style="color:#ff7f0e">{n_unc} uncertain</span>
            </div>
            {fmt_rubric_labels(rubrics, RUBRIC_ITEMS)}
        </div>
    </div>
    '''
    display(HTML(html))

print(f'\nTotal: {len(all_rows_sorted)} examples')

In [ ]:
# -----------------------------
# Select and save examples by index from gallery (manual selection)
# -----------------------------
# Edit SELECTED_INDICES to pick examples by their # from the gallery above
SELECTED_FILES = ['129_69650.jpg', '154_33500.jpg', '152_41050.jpg', '148_21675.jpg']
SELECTED_INDICES = [next(i + 1 for i, r in enumerate(all_rows_sorted) if r['file_name'] == name) for name in SELECTED_FILES]
EXPORT = False

selected_rows_manual = [all_rows_sorted[i - 1] for i in SELECTED_INDICES]

OUT_SINGLES_DIR = REPO_ROOT / 'few_shot_examples' / 'endoscapes' / 'rubrics' / 'filtered'
OUT_SINGLES_DIR.mkdir(parents=True, exist_ok=True)
jsonl_path = OUT_SINGLES_DIR / 'selected_examples_v3.jsonl'
image_dir = OUT_SINGLES_DIR / 'images'
image_dir.mkdir(parents=True, exist_ok=True)

records_manual = []
if EXPORT:
    with jsonl_path.open('w', encoding='utf-8') as f:
        for r in selected_rows_manual:
            gt = r['gt']
            rubrics = r.get('rubrics', {})
            gp = gt_pattern(gt)
            n_yes = sum(1 for v in rubrics.values() if v == 'yes')
            n_no = sum(1 for v in rubrics.values() if v == 'no')
            rec = {
                'file_name': r['file_name'],
                'image_path': r.get('image_path'),
                'video_id': r.get('video_id'),
                'frame_id': r.get('frame_id'),
                'split': r.get('split'),
                'gt': gt,
                'rubric_yes': n_yes,
                'rubric_no': n_no,
                'rubric_total': len(rubrics),
                'gt_pattern': gp,
                'rubrics': rubrics,
            }
            src = LABELS_ROOT / 'val' / rec['file_name']
            out_name = rec['file_name']
            if src and src.exists():
                dst = image_dir / out_name
                shutil.copy2(src, dst)
                rec['saved_image'] = dst.relative_to(REPO_ROOT).as_posix()
                rec['image_path'] = (Path('data/endoscapes/val') / rec['file_name']).as_posix()
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            records_manual.append(rec)
    
print(f'Saved {len(records_manual)} examples to {jsonl_path}' if EXPORT else 'Export disabled; displaying the four paper examples.')

# Display with rubric detail
display(HTML('<h3>Manually selected examples (singles)</h3>'))
for sel_idx, r in zip(SELECTED_INDICES, selected_rows_manual):
    gt = r['gt']
    rubrics = r.get('rubrics', {})
    gp = gt_pattern(gt)
    vid = r.get('video_id', '')
    img_path = str(LABELS_ROOT / 'val' / r['file_name'])
    n_yes = sum(1 for v in rubrics.values() if v == 'yes')
    n_no = sum(1 for v in rubrics.values() if v == 'no')
    n_unc = sum(1 for v in rubrics.values() if v == 'uncertain')
    gt_parts = []
    for c in ['c1', 'c2', 'c3']:
        v = gt.get(c, 0)
        color = '#2ca02c' if v == 1 else '#d62728'
        gt_parts.append(f'<span style="color:{color};font-weight:bold">{c.upper()}={v}</span>')
    gt_html = ' &nbsp; '.join(gt_parts)
    html = f'''
    <div style="border:2px solid #8e44ad;padding:10px;margin:8px 0;display:flex;gap:15px;align-items:flex-start">
        <div style="flex-shrink:0">
            <div style="font-weight:bold;font-size:13px;margin-bottom:4px">
                #{sel_idx} &nbsp; GT: {gp}
            </div>
            {img_tag(img_path, width=350)}
            <div style="font-size:11px;color:#666;margin-top:4px">
                video: {vid}<br>file: {r['file_name']}
            </div>
        </div>
        <div style="flex:1">
            <div style="margin-bottom:6px">
                <b>GT:</b> {gt_html} &nbsp;&nbsp;
                <b>Rubrics:</b> <span style="color:#2ca02c">{n_yes} yes</span>,
                <span style="color:#d62728">{n_no} no</span>,
                <span style="color:#ff7f0e">{n_unc} uncertain</span>
            </div>
            {fmt_rubric_labels(rubrics, RUBRIC_ITEMS)}
        </div>
    </div>'''
    display(HTML(html))